# 05_遷移學習_MobileNet.py
**電腦視覺與深度學習 › Day18 CNN + Teachable Machine + MediaPipe**

這支程式直接借用 Google 在 ImageNet 上訓練好的 MobileNetV2,不用自己訓練,就能認出一千種常見物體。這就是遷移學習的力量。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/4/2)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> ⚡ 這支會用到深度學習,建議先到「執行階段 › 變更執行階段類型」選 **T4 GPU**。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY9"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '05_遷移學習_MobileNet.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    if not os.path.exists('/content/_sample.jpg'):
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', '/content/_sample.jpg')
    return cv2.imread('/content/_sample.jpg')
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 什麼是遷移學習
註解說明:別人已經花大量 GPU 訓練好模型,我們直接拿來用。Teachable Machine 的底層也是這招:沿用 MobileNet 的特徵,只重新訓練最後一層。

In [ ]:
# 05 — 用預訓練 MobileNetV2 分類 webcam 畫面

### 匯入
從 Keras 的應用模組匯入 MobileNetV2,以及配套的前處理和解讀結果的函式。

In [ ]:
#
# 觀念：遷移學習 (Transfer Learning)
#   Google 已經花很多 GPU 在 ImageNet（120 萬張，1000 類）訓練好 MobileNetV2
#   我們直接拿來用 —— 不用重訓就能認出 1000 種常見物體（狗、貓、水果、家電...）
#
# 這就是 Teachable Machine 的底層策略：拿 MobileNet 當「特徵萃取器」，
# 你只重訓最後一層來認你的類別。
#
# 需要：pip install tensorflow opencv-python numpy

import cv2
import numpy as np
import tensorflow as tf

### 載入預訓練權重
weights 等於 imagenet,第一次執行會下載大約 14 MB 的權重,之後就存在電腦裡。

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import (
    MobileNetV2, preprocess_input, decode_predictions,
)

# ====== 載入 ImageNet 預訓練權重 ======
# 第一次跑會下載約 14MB 權重，之後會 cache 在 ~/.keras/
print("載入預訓練 MobileNetV2 ...")
model = MobileNetV2(weights="imagenet")

### 開鏡頭
打開鏡頭,打不開就丟出錯誤。

In [ ]:
print("完成！模型輸入：", model.input_shape)

# ====== 開 webcam ======
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("無法開啟 webcam")

print("按 q 離開；把物體對著鏡頭讓它認")

### 前處理
縮成 224 乘 224、轉成 RGB,再交給 MobileNet 自己的 preprocess_input。它和 Teachable Machine 的除以 127.5 減 1 是一模一樣的運算。

### 推論並取前幾名
predict 得到一千類的機率,decode_predictions 幫我們翻成英文類別名稱,只留前 3 名。

### 寫在畫面上
把前幾名的名稱和百分比一行一行寫在畫面左上角,再顯示出來,按 q 離開。

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 預處理：MobileNetV2 要 224x224 RGB，用它自帶的 preprocess_input
    img = cv2.resize(frame, (224, 224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    img = preprocess_input(img)
    img = np.expand_dims(img, axis=0)

    # 推論 → 取前 3 名
    preds = model.predict(img, verbose=0)
    top3 = decode_predictions(preds, top=3)[0]
    # top3 = [(imagenet_id, 名字, 機率), ...]

    # 畫到畫面上
    for i, (_, name, score) in enumerate(top3):
        text = f"{i+1}. {name}: {score*100:.1f}%"
        cv2.putText(frame, text, (10, 40 + i * 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow("MobileNetV2 (ImageNet 1000 class)", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

### 收拾與小提示
放掉鏡頭。最後的小提示提醒你:這裡完全沒有 fit,因為權重是別人訓練好的;想認自己的類別,就用 Teachable Machine 或自己做微調。

In [ ]:
cap.release()
cv2.destroyAllWindows()

# ====== 小提示 ======
# 這裡「model.predict」就是「推論」；我們完全沒有 model.fit，
# 因為權重已經是別人訓練好的。
#
# 想要它認你的類別（不在 ImageNet 1000 類裡）怎麼辦？
#   A. 用 Teachable Machine（GUI 遷移學習）
#   B. 自己寫 fine-tune：把 top 拿掉，接自己的 Dense 層，只訓練新加的層
#      → 進階練習可以參考 tf.keras Application 官方教學

## 小結
你看到預訓練模型不用訓練也能用。下一支作業,要把你自己的 Teachable Machine 模型做成有用的應用。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/4/2) 看逐句拆解、做練習。